In [1]:
import json
import base64
from openai import OpenAI

In [2]:
client = OpenAI(
    api_key="os.environ.get("OPENAI_API_KEY")"
)

In [3]:
SYSTEM_PROMPT = """\
You are an answer-matching evaluator.

Your task is to compare two answers and decide whether they have the same meaning.

The answers do not need to use the exact same words. They should be considered a match if they express the same core idea, refer to the same entity, or would be accepted as semantically equivalent in context.

You should focus on meaning, not wording.

Return only one of the following labels:

MATCH
NO_MATCH

Guidelines:
- Mark MATCH if both answers convey the same meaning, even with different wording.
- Mark MATCH if one answer is more detailed but does not contradict the other.
- Mark MATCH for synonyms, paraphrases, abbreviations, or equivalent medical/scientific terms.
- Mark NO_MATCH if the answers refer to different entities, objects, actions, locations, numbers, or concepts.
- Mark NO_MATCH if one answer contradicts the other.
- Mark NO_MATCH if one answer is too vague to confirm equivalence.
- Ignore capitalization, punctuation, grammar mistakes, and minor wording differences.

Input format:
Answer 1: <first answer>
Answer 2: <second answer>

Output format:
MATCH or NO_MATCH
"""

In [4]:
def _parse_json(raw: str) -> dict:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.lower().startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())

In [5]:
def _encode_image(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

In [6]:
def compare_answers(answer1: str, answer2: str) -> str:
    """Compare two answers and return whether they match in meaning.
    
    Returns: "MATCH" or "NO_MATCH"
    """
    
    user_text = (
        f"Answer 1: {answer1}\n"
        f"Answer 2: {answer2}"
    )
    
    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=SYSTEM_PROMPT,
            input=user_text,
        )
        
        result = response.output_text.strip()
        
        # Extract MATCH or NO_MATCH from response
        if "MATCH" in result.upper():
            if "NO_MATCH" in result.upper():
                # If both are present, check which comes first or appears more prominently
                no_match_idx = result.upper().find("NO_MATCH")
                match_idx = result.upper().find("MATCH")
                if no_match_idx < match_idx:
                    return "NO_MATCH"
            return "MATCH"
        else:
            return "NO_MATCH"
            
    except Exception as e:
        print(f"Error comparing answers: {e}")
        return None


In [10]:
import json

with open('/home/as5606/Datasets/Cholec_text_semantic_similar_questions/hulu_med_4B_semantic_variations_answers.json', 'r') as f:
    hulumed_4B_semantic_variations_answers = json.load(f)

In [11]:
hulumed_4B_semantic_variations_answers[0]

{'id': 'video31_frame001200',
 'image': '/home/as5606/Datasets/chole80_framed/cholec80/frames/video31/video31_000049.png',
 'original_question': 'is scissors used in preparation?',
 'original_answer': 'scissors is not used in preparation',
 'semantic_variations_with_hulu_answers': [{'question': 'During preparation, is scissors used?',
   'hulu_med_answer': 'No'},
  {'question': 'Is scissors employed in the preparation phase?',
   'hulu_med_answer': 'No. There are no scissors visible in the image. The only surgical instrument shown is a grasper or forceps.'},
  {'question': 'Does preparation involve the use of scissors?',
   'hulu_med_answer': 'No. There are no scissors visible in the image. The only surgical instrument shown is a grasper or forceps.'},
  {'question': 'In preparation, is scissors being used?',
   'hulu_med_answer': 'No'},
  {'question': 'Is the scissors used during preparation?',
   'hulu_med_answer': 'Yes'}]}